In [1]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
from ib_insync import util, Contract
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

IBKR connected: True


Error 162, reqId 4: Historical Market Data Service error message:No data of type EODChart is available for the exchange 'CME' and the security type 'Futures Options' and '30 d' and '1 day', contract: Contract(secType='FOP', conId=751506503, symbol='BRR', lastTradeDateOrContractMonth='20260626', strike=1000.0, right='C', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCM6 C1000', tradingClass='BTC')
Error 162, reqId 5: Historical Market Data Service error message:No data of type EODChart is available for the exchange 'CME' and the security type 'Futures Options' and '30 d' and '1 day', contract: Contract(secType='FOP', conId=751506503, symbol='BRR', lastTradeDateOrContractMonth='20260626', strike=1000.0, right='C', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCM6 C1000', tradingClass='BTC')
Error 162, reqId 6: Historical Market Data Service error message:No data of type EODChart is available for the exchange 'CME' and the security type 'Futures Options' 

In [3]:
import pandas as pd

conId_df = pd.read_csv("options_conIds.csv")
conId_df.head()

,conId_call,expiry,strike,conId_put
0,751506503,20260626,1000.0,751506545
1,751506548,20260626,5000.0,751506518
2,751506561,20260626,10000.0,751506558
3,852657225,20260626,17500.0,852657015
4,764688659,20260626,20000.0,764688665


In [4]:
lookback_period = "30 D"
bar_size = "1 day"
use_regular_trading_hours = False

In [5]:
prices_to_use = "MIDPOINT"

'''

    "TRADES"
    "MIDPOINT"
    "BID"
    "ASK"
    "BID_ASK"
    "ADJUSTED_LAST"
    "HISTORICAL_VOLATILITY"
    "OPTION_IMPLIED_VOLATILITY"
    "FEE_RATE"
    "REBATE_RATE"
    "SCHEDULE"
        
'''

'\n\n    "TRADES"\n    "MIDPOINT"\n    "BID"\n    "ASK"\n    "BID_ASK"\n    "ADJUSTED_LAST"\n    "HISTORICAL_VOLATILITY"\n    "OPTION_IMPLIED_VOLATILITY"\n    "FEE_RATE"\n    "REBATE_RATE"\n    "SCHEDULE"\n        \n'

In [6]:
for conId in conId_df.conId_call:
    # Build contract directly from conId
    option_contract = Contract(
        conId=int(conId),
        exchange="CME",
        currency="USD"
    )

    # Qualify/fill contract details
    qualified = await ibkr.ib.qualifyContractsAsync(option_contract)
    if not qualified:
        print(f"Could not qualify conId {conId}")
        continue

    option_contract = qualified[0]

    for what_to_show in ["TRADES", "MIDPOINT", "BID_ASK"]:
        try:
            bars = await ibkr.ib.reqHistoricalDataAsync(
                contract=option_contract,
                endDateTime="",
                durationStr=lookback_period,
                barSizeSetting=bar_size,
                whatToShow=what_to_show,
                useRTH=False,
                formatDate=1,
                keepUpToDate=False
            )

            df = util.df(bars)

            if df is not None and not df.empty:
                df["conId"] = int(conId)
                df["localSymbol"] = option_contract.localSymbol
                df["whatToShow"] = what_to_show
                all_rows.append(df)

                print(f"Got {len(df)} bars for {conId} using {what_to_show}")
                break
            else:
                print(f"No data for {conId} using {what_to_show}")

        except Exception as e:
            print(f"Failed {conId} using {what_to_show}: {e}")

historical_options_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

historical_options_df.to_csv("historical_option_prices.csv", index=False)

No data for 751506503 using TRADES
No data for 751506503 using MIDPOINT
No data for 751506503 using BID_ASK
No data for 751506548 using TRADES
No data for 751506548 using MIDPOINT
No data for 751506548 using BID_ASK
No data for 751506561 using TRADES
No data for 751506561 using MIDPOINT
No data for 751506561 using BID_ASK
No data for 852657225 using TRADES
No data for 852657225 using MIDPOINT
No data for 852657225 using BID_ASK
No data for 764688659 using TRADES
No data for 764688659 using MIDPOINT
No data for 764688659 using BID_ASK
No data for 852140904 using TRADES
No data for 852140904 using MIDPOINT
No data for 852140904 using BID_ASK
No data for 816142851 using TRADES
No data for 816142851 using MIDPOINT
No data for 816142851 using BID_ASK
No data for 852140794 using TRADES
No data for 852140794 using MIDPOINT
No data for 852140794 using BID_ASK
No data for 764688940 using TRADES
No data for 764688940 using MIDPOINT
No data for 764688940 using BID_ASK
No data for 852141169 using 

NameError: name 'all_rows' is not defined